# E-Commerce Customer Analytics: RFM Segmentation

Author: Nidhi Sharma

This notebook takes the cleaned transactions from SQL Server (`dbo.RetailCleaned`), scores every customer on **Recency, Frequency and Monetary** value, assigns a business segment and a 90-day churn flag, and exports the result for the Power BI dashboard.

## 1. Load and prepare the cleaned data

In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('Retailcleaned.csv',header=None,low_memory=False)

In [3]:
df.columns=['Invoice','StockCode','Description','Quantity','InvoiceDate','Price','Customer_ID','Country','SourceSheet']

In [4]:
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])

In [5]:
print(df.shape)

(1055238, 9)


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1055238 entries, 0 to 1055237
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1055238 non-null  object        
 1   StockCode    1055238 non-null  object        
 2   Description  1050856 non-null  object        
 3   Quantity     1055238 non-null  int64         
 4   InvoiceDate  1055238 non-null  datetime64[ns]
 5   Price        1055233 non-null  float64       
 6   Customer_ID  812368 non-null   float64       
 7   Country      1055238 non-null  object        
 8   SourceSheet  1055238 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 72.5+ MB


In [7]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,SourceSheet
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2010-2011
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,2010-2011
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2010-2011
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2010-2011
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,2010-2011


## 2. Calculate Recency, Frequency and Monetary

Only rows with a valid `Customer_ID` and a positive `Quantity` are used. Snapshot date: 2011-12-10.

In [8]:
# Filter and calculate recency, frequency and monetary value
rfm_data = df[
    (df['Customer_ID'].notna()) &
    (df['Quantity'] > 0)
].copy()

In [9]:
print(rfm_data.shape)
print(rfm_data['Customer_ID'].isna().sum())
print(rfm_data['Quantity'].min())

(793680, 9)
0
1


In [10]:
snapshot_date=pd.to_datetime('2011-12-10')

In [11]:
rfm=rfm_data.groupby('Customer_ID').agg(
    Recency=('InvoiceDate',lambda x: (snapshot_date-x.max()).days),
    Frequency=('Invoice','nunique'),
    Monetary=('Quantity',lambda x:(x*rfm_data.loc[x.index,'Price']).sum())
).reset_index()

In [12]:
print(rfm.head())
print(rfm.describe())
print(rfm.isnull().sum())

   Customer_ID  Recency  Frequency  Monetary
0      12346.0      325         12  77556.46
1      12347.0        2          8   5633.32
2      12348.0       75          5   2019.40
3      12349.0       18          4   4428.69
4      12350.0      310          1    334.40
        Customer_ID      Recency    Frequency       Monetary
count   5881.000000  5881.000000  5881.000000    5881.000000
mean   15314.674205   200.992858     6.287366    3007.219966
std     1715.429759   209.489651    13.012973   14728.322206
min    12346.000000     0.000000     1.000000       0.000000
25%    13833.000000    25.000000     1.000000     344.750000
50%    15313.000000    95.000000     3.000000     886.570000
75%    16797.000000   379.000000     7.000000    2297.660000
max    18287.000000   738.000000   398.000000  608821.650000
Customer_ID    0
Recency        0
Frequency      0
Monetary       0
dtype: int64


## 3. Score customers (1-5 quintiles) and assign segments

In [13]:
# Score customers and assign business segments

rfm['R_score'] = pd.qcut(
    rfm['Recency'],
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm['F_score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm['M_score'] = pd.qcut(
    rfm['Monetary'],
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

In [14]:
print(rfm[['Customer_ID', 'R_score', 'F_score', 'M_score']].head(10))

print(rfm[['R_score', 'F_score', 'M_score']].describe())

   Customer_ID  R_score  F_score  M_score
0      12346.0        2        5        5
1      12347.0        5        4        5
2      12348.0        3        4        4
3      12349.0        5        3        5
4      12350.0        2        1        2
5      12351.0        2        1        2
6      12352.0        4        5        4
7      12353.0        2        2        2
8      12354.0        2        1        3
9      12355.0        2        2        3
           R_score      F_score      M_score
count  5881.000000  5881.000000  5881.000000
mean      3.009692     2.999660     2.999660
std       1.420121     1.414454     1.414454
min       1.000000     1.000000     1.000000
25%       2.000000     2.000000     2.000000
50%       3.000000     3.000000     3.000000
75%       4.000000     4.000000     4.000000
max       5.000000     5.000000     5.000000


In [15]:
def segment_customer(row):
    if  row['R_score']>=4 and row['F_score']>=4 and row['M_score']>=4:
        return 'Champions'
    elif row['R_score']>=3 and row['F_score']>=3:
        return 'Loyal'
    elif row['R_score']<=2 and row['F_score']>=3:
        return 'At risk'
    elif row['R_score']>=4 and row['F_score']<=2:
        return 'New'
    elif row['R_score']==3 and row['F_score']<=2:
        return 'Needs attention'
    else:
        return 'Lost'
rfm['Segment']=rfm.apply(segment_customer,axis=1)
rfm['IsChurned']=rfm['Recency']>90

## 4. Segment and churn summary

In [16]:
# Count customers in each segment
print(rfm['Segment'].value_counts())

Segment
Lost               1524
Loyal              1414
Champions          1293
At risk             821
New                 442
Needs attention     387
Name: count, dtype: int64


In [17]:
# Calculate churned and active customers
print(rfm['IsChurned'].value_counts())

IsChurned
True     2991
False    2890
Name: count, dtype: int64


In [18]:
# Average monetary value by segment
print(rfm.groupby('Segment')['Monetary'].mean().sort_values(ascending=False))

Segment
Champions          9335.826628
At risk            1978.306541
Loyal              1929.334767
New                 889.204389
Needs attention     530.889535
Lost                435.357723
Name: Monetary, dtype: float64


## 5. Merge RFM results back onto transactions and export for Power BI

In [19]:
# Merge rfm results back onto transactions and export
df = df[
    (df['Customer_ID'].notna()) &
    (df['Quantity'] > 0)
].copy()

In [20]:
# Create invoice month column
df['InvoiceMonth'] = (
    df['InvoiceDate']
    .dt.to_period('M')
    .dt.to_timestamp()
)

# Calculate revenue
df['Revenue'] = df['Quantity'] * df['Price']

In [21]:
print("df rows:", len(df))
print("rfm rows:", len(rfm))

df rows: 793680
rfm rows: 5881


In [22]:
import gc

rfm_small = rfm[
    ['Customer_ID', 'Recency', 'Frequency', 'Monetary',
     'R_score', 'F_score', 'M_score', 'Segment', 'IsChurned']
]

# Check duplicate customer IDs
print("Unique customers:", rfm_small['Customer_ID'].is_unique)

chunk_size = 50000
output_file = 'EcommerceDashboardData.csv'

for start in range(0, len(df), chunk_size):

    chunk = df.iloc[start:start + chunk_size]

    merged_chunk = chunk.merge(
        rfm_small,
        on='Customer_ID',
        how='left',
        validate='many_to_one'
    )

    merged_chunk.to_csv(
        output_file,
        mode='w' if start == 0 else 'a',
        header=(start == 0),
        index=False
    )

    print(f"Completed: {min(start + chunk_size, len(df))} / {len(df)}")

    del merged_chunk
    gc.collect()

print("Export completed successfully!")

Unique customers: True
Completed: 50000 / 793680
Completed: 100000 / 793680
Completed: 150000 / 793680
Completed: 200000 / 793680
Completed: 250000 / 793680
Completed: 300000 / 793680
Completed: 350000 / 793680
Completed: 400000 / 793680
Completed: 450000 / 793680
Completed: 500000 / 793680
Completed: 550000 / 793680
Completed: 600000 / 793680
Completed: 650000 / 793680
Completed: 700000 / 793680
Completed: 750000 / 793680
Completed: 793680 / 793680
Export completed successfully!


## 6. Check the exported file

In [23]:
import os

print("File exists:", os.path.exists(output_file))
print("File size (MB):", round(os.path.getsize(output_file) / (1024**2), 2))

File exists: True
File size (MB): 119.66


In [24]:
df_check = pd.read_csv(output_file, nrows=5)

print(df_check.head())
print(df_check.columns.tolist())

   Invoice  StockCode                         Description  Quantity  \
0   489434      21232      STRAWBERRY CERAMIC TRINKET BOX        24   
1   489434      21523  FANCY FONT HOME SWEET HOME DOORMAT        10   
2   489434      21871                 SAVE THE PLANET MUG        24   
3   489434      22041        RECORD FRAME 7" SINGLE SIZE         48   
4   489434      22064          PINK DOUGHNUT TRINKET POT         24   

           InvoiceDate  Price  Customer_ID         Country SourceSheet  \
0  2009-12-01 07:45:00   1.25      13085.0  United Kingdom   2010-2011   
1  2009-12-01 07:45:00   5.95      13085.0  United Kingdom   2010-2011   
2  2009-12-01 07:45:00   1.25      13085.0  United Kingdom   2010-2011   
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom   2010-2011   
4  2009-12-01 07:45:00   1.65      13085.0  United Kingdom   2010-2011   

  InvoiceMonth  Revenue  Recency  Frequency  Monetary  R_score  F_score  \
0   2009-12-01     30.0      157          8   2433.28